In [0]:
-- =============================================================================
-- SECTION 2: PIPELINE HEALTH
-- =============================================================================

-- PL-1  Row counts per layer — all time
-- Widget: Table  (simple health snapshot; latest_event confirms each layer is live)
-- ─────────────────────────────────────────────────────────────────────────────
SELECT 'bronze'            AS layer, COUNT(*) AS total_rows, MAX(ingest_timestamp) AS latest_event
FROM wiki_poc.poc.bronze_recentchange_raw
UNION ALL
SELECT 'silver_enwiki',              COUNT(*), MAX(ingest_timestamp)
FROM wiki_poc.poc.silver_recentchange_enwiki
UNION ALL
SELECT 'silver_quarantine',          COUNT(*), MAX(ingest_timestamp)
FROM wiki_poc.poc.silver_recentchange_quarantine
UNION ALL
SELECT 'gold_5min_windows',          COUNT(*), MAX(window_end)
FROM wiki_poc.poc.gold_page_edits_5min;


-- PL-2  Silver pass rate and quarantine rate — last hour
-- Widget: Counter  |  values=pass_rate_pct, quarantine_rate_pct
-- Expected: pass_rate_pct ~5–15%, quarantine_rate_pct ~85–95%
-- ─────────────────────────────────────────────────────────────────────────────
SELECT
  b.bronze_1h,
  s.silver_1h,
  q.quarantine_1h,
  ROUND(100.0 * s.silver_1h    / NULLIF(b.bronze_1h, 0), 1) AS pass_rate_pct,
  ROUND(100.0 * q.quarantine_1h / NULLIF(b.bronze_1h, 0), 1) AS quarantine_rate_pct
FROM
  (SELECT COUNT(*) AS bronze_1h      FROM wiki_poc.poc.bronze_recentchange_raw       WHERE ingest_timestamp >= NOW() - INTERVAL 1 HOUR) b,
  (SELECT COUNT(*) AS silver_1h      FROM wiki_poc.poc.silver_recentchange_enwiki     WHERE ingest_timestamp >= NOW() - INTERVAL 1 HOUR) s,
  (SELECT COUNT(*) AS quarantine_1h  FROM wiki_poc.poc.silver_recentchange_quarantine WHERE ingest_timestamp >= NOW() - INTERVAL 1 HOUR) q;


-- PL-3  Quarantine breakdown by reason — last hour
-- Widget: Bar chart  |  x=quarantine_reason  y=events (sorted desc)
-- Expected dominant reason: non_enwiki (most of the firehose), then non_edit_type, then bot
-- ─────────────────────────────────────────────────────────────────────────────
SELECT
  quarantine_reason,
  COUNT(*)                                              AS events,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)  AS pct
FROM wiki_poc.poc.silver_recentchange_quarantine
WHERE ingest_timestamp >= NOW() - INTERVAL 1 HOUR
GROUP BY quarantine_reason
ORDER BY events DESC;


-- PL-4  Silver events per 5-minute bucket — last 2 hours
-- Widget: Line chart  |  x=bucket_start  y=silver_events
-- Gaps in this chart (buckets with 0 or no row) indicate pipeline stalls
-- ─────────────────────────────────────────────────────────────────────────────
SELECT
  from_unixtime(FLOOR(unix_timestamp(ingest_timestamp) / 300) * 300)  AS bucket_start,
  COUNT(*)                                                              AS silver_events
FROM wiki_poc.poc.silver_recentchange_enwiki
WHERE ingest_timestamp >= NOW() - INTERVAL 2 HOURS
GROUP BY 1
ORDER BY 1;


-- PL-5  Gold windows landing — last 2 hours
-- Widget: Line chart  |  x=window_start  y=total_edits
-- Windows arrive ~10-15 min after they close (watermark lag) — a flat line
-- means the watermark is advancing but no edits are landing (unusual)
-- ─────────────────────────────────────────────────────────────────────────────
SELECT
  window_start,
  SUM(edit_count)        AS total_edits,
  COUNT(DISTINCT title)  AS pages_with_edits
FROM wiki_poc.poc.gold_page_edits_5min
WHERE window_start >= NOW() - INTERVAL 2 HOURS
GROUP BY window_start
ORDER BY window_start;
